# Breadth Data — Exploration & Report Development

Scratch notebook for developing the market-breadth monitoring report.

Data is produced by `GetFMPData/breadth_data_pull.py` (docs: `documentation/breadth-universe-task.md`):
- **qualified ticker list** — `GetFMPData/universe/breadth-qualified-<label>.csv`
- **liquidity panel** (15 months of adj/raw prices + EV fields + 20d liquidity metrics) — `s3://…/data_price-ev-liquidity_<label>.parquet`

Liquidity definition lives in `util/dataset_builder/merge_core.py :: add_liquidity_flags`.
To refresh after tuning it: `python GetFMPData/breadth_data_pull.py --skip-pull --label <label> --overwrite`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # DevLab/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from util.dataset_builder.s3_io import (
    dataset_base_path, get_s3_filesystem, load_single_parquet,
)

pd.set_option('display.max_columns', 100)

In [ ]:
LABEL = '20260612-breadth'

# default panel columns for breadth work; pass columns=None to load everything
PANEL_COLS = [
    'symbol', 'date', 'exchange', 'sector',
    'adjOpen', 'adjHigh', 'adjLow', 'adjClose', 'adjVolume',
    'rawClose', 'rawVolume', 'dollarVolume',
    'price_tr20', 'dollarVolume_tr20', 'turnOver_tr20',
    'marketCapitalization', 'lowLiquidity',
]

In [ ]:
fs = get_s3_filesystem()


def load_qualified(label=LABEL):
    """Qualified ticker list written by breadth_data_pull.py."""
    path = REPO_ROOT / 'GetFMPData' / 'universe' / f'breadth-qualified-{label}.csv'
    return pd.read_csv(path, parse_dates=['asOfDate'])


def load_panel(label=LABEL, columns=PANEL_COLS, symbols=None):
    """Liquidity panel from S3 (~1.9M rows for the full active universe).

    Returns rows sorted by (symbol, date) — ready for util/features.
    """
    df = load_single_parquet(fs, f'data_price-ev-liquidity_{label}.parquet',
                             columns=columns, base_path=dataset_base_path())
    if symbols is not None:
        df = df[df['symbol'].isin(set(symbols))]
    return df.sort_values(['symbol', 'date']).reset_index(drop=True)

In [ ]:
qualified = load_qualified()
print(f'{len(qualified):,} qualified symbols')
qualified.head()

In [ ]:
panel = load_panel(symbols=qualified['symbol'])
print(f"{len(panel):,} rows, {panel['symbol'].nunique():,} symbols, "
      f"{panel['date'].min():%Y-%m-%d} -> {panel['date'].max():%Y-%m-%d}")
panel.head()

## Exploration (scratch)

Starter example: % of the qualified universe above its 50-day moving average.

In [ ]:
ma50 = panel.groupby('symbol')['adjClose'].transform(lambda s: s.rolling(50).mean())
valid = ma50.notna()
pct_above = ((panel['adjClose'] > ma50)[valid]
             .groupby(panel.loc[valid, 'date']).mean())

fig, ax = plt.subplots(figsize=(10, 3))
pct_above.plot(ax=ax)
ax.set_title('% of qualified universe above 50-day MA')
ax.axhline(0.5, color='grey', lw=0.5)
plt.show()

## Report development (placeholder)

Candidate breadth metrics to build out (then graduate to a `util/` module + entry point):
- % above 20 / 50 / 200-day MA
- advance / decline line, up- vs down-volume
- new 63d / 252d highs and lows
- breadth by sector / exchange
- reuse `util/features` primitives where they fit (panel already satisfies the symbol/date contract)

Output target (suggested): `reports/market_breadth/<date>/`.